# FedAvg — IID Federated Baseline, Optimized V2

This notebook runs the **IID FedAvg experiment** using the artifact produced by `fl-dataset-preparation-optimized-v2.ipynb`.

The model/training choices are intentionally aligned with the completed centralized baseline:

- MLP: **39 → 128 → 64 → 34**
- optimizer: **Adam**
- learning rate: **0.001**
- batch size: **8192**
- loss: **CrossEntropyLoss**
- class weights: **off**
- AMP: **on when CUDA is available**
- centralized baseline training budget: **3 epochs**
- FL budget here: **3 rounds × 1 local epoch**, with all 10 clients participating each round

Because every client participates in every round, `3 rounds × 1 local epoch` makes the full training split pass through local training three times overall, giving a simple training-exposure match to the centralized 3-epoch baseline.

## Kaggle input for this notebook

Do **not** add the raw CICIoT2023 dataset again.

First run and save `fl-dataset-preparation-optimized-v2.ipynb`. Then, in this notebook:

1. Click **Add Input**.
2. Add the saved output of your dataset-preparation notebook.
3. Enable **GPU** (Tesla T4 or better recommended).
4. Run all cells.

The code automatically searches `/kaggle/input` for the correct prepared artifact by reading its `metadata.json`, so the exact Kaggle folder slug does not need to be hard-coded.


In [ ]:
# Cell 02: Imports
from pathlib import Path
from contextlib import nullcontext
import copy
import gc
import json
import os
import platform
import random
import time
import zipfile

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)

import torch
import torch.nn as nn


In [ ]:
# Cell 03: Configuration — aligned with centralized-baseline-markp.ipynb
EXPERIMENT_NAME = "fedavg_iid_optimized_v2"
EXPECTED_ARTIFACT_TYPE = "fl_prepared_iid_v2"

SEED = 42
NUM_ROUNDS = 3
LOCAL_EPOCHS = 1
BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 32768
LEARNING_RATE = 0.001
HIDDEN_1 = 128
HIDDEN_2 = 64

USE_CLASS_WEIGHTS = False
USE_AMP = True
KEEP_TRAIN_ON_GPU = True
GPU_DATA_MAX_FRACTION = 0.70
RESUME_FROM_CHECKPOINT = True

OUTPUT_DIR = Path("/kaggle/working/fedavg_iid_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "fedavg_iid_checkpoint.pt"

print("Experiment:", EXPERIMENT_NAME)
print("Rounds:", NUM_ROUNDS)
print("Local epochs per round:", LOCAL_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)


In [ ]:
# Cell 04: Reproducibility, device, and RAM helper
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def show_ram(prefix="RAM"):
    process_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    system = psutil.virtual_memory()
    print(f"{prefix} | Process: {process_gb:.2f} GB | System used: {system.percent:.1f}%")


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_b/1024**3:.2f} GB free / {total_b/1024**3:.2f} GB total")
else:
    print("WARNING: GPU is strongly recommended for the full-data experiment.")

show_ram("Startup")


In [ ]:
# Cell 05: Automatically locate the prepared IID artifact in Kaggle inputs
def find_prepared_artifact(root=Path("/kaggle/input")):
    matches = []
    for metadata_path in root.rglob("metadata.json"):
        try:
            with open(metadata_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            if meta.get("artifact_type") == EXPECTED_ARTIFACT_TYPE:
                matches.append((metadata_path.parent, meta))
        except Exception:
            continue

    if not matches:
        raise FileNotFoundError(
            "Prepared FL artifact not found. First run and Save Version of "
            "fl-dataset-preparation-optimized-v2.ipynb, then Add Input -> that notebook output."
        )

    if len(matches) > 1:
        print("Multiple prepared artifacts found; using the first one:")
        for path, _ in matches:
            print("-", path)

    return matches[0]


DATA_DIR, metadata = find_prepared_artifact()

print("Prepared artifact:", DATA_DIR)
print("Dataset fraction:", metadata["dataset_frac_requested"])
print("Train samples:", f"{metadata['train_samples']:,}")
print("Test samples:", f"{metadata['test_samples']:,}")
print("Clients:", metadata["num_clients"])
print("Partition:", metadata["partition"])

if metadata["seed"] != SEED:
    raise ValueError(f"Seed mismatch: artifact={metadata['seed']} notebook={SEED}")


In [ ]:
# Cell 06: Load metadata, indices, and memory-mapped feature/label arrays
load_start = time.perf_counter()

with open(DATA_DIR / "classes.json", "r", encoding="utf-8") as f:
    classes = json.load(f)
with open(DATA_DIR / "feature_cols.json", "r", encoding="utf-8") as f:
    feature_cols = json.load(f)

X_mm = np.load(DATA_DIR / "X.npy", mmap_mode="r")
y_mm = np.load(DATA_DIR / "y.npy", mmap_mode="r")
train_idx = np.load(DATA_DIR / "train_idx.npy")
test_idx = np.load(DATA_DIR / "test_idx.npy")
scaler_mean = np.load(DATA_DIR / "scaler_mean.npy").astype(np.float32)
scaler_scale = np.load(DATA_DIR / "scaler_scale.npy").astype(np.float32)

NUM_CLIENTS = int(metadata["num_clients"])
client_positions = [
    np.load(DATA_DIR / f"client_{i:02d}_positions.npy")
    for i in range(NUM_CLIENTS)
]

assert X_mm.shape[1] == 39
assert len(feature_cols) == 39
assert len(classes) == 34
assert scaler_mean.shape == (39,)
assert scaler_scale.shape == (39,)
assert sum(len(x) for x in client_positions) == len(train_idx)

print("X mmap:", X_mm.shape, X_mm.dtype)
print("y mmap:", y_mm.shape, y_mm.dtype)
print("Client sizes:", [len(x) for x in client_positions])
show_ram("After mmap loading")


In [ ]:
# Cell 07: Materialize train/test arrays once and apply the saved StandardScaler
# This mirrors the optimized centralized baseline: pay the CPU indexing/scaling cost once,
# then keep training batches in PyTorch instead of repeatedly indexing the mmap file.
materialize_start = time.perf_counter()

X_train = np.ascontiguousarray(X_mm[train_idx], dtype=np.float32)
y_train = np.ascontiguousarray(y_mm[train_idx], dtype=np.int64)
X_test = np.ascontiguousarray(X_mm[test_idx], dtype=np.float32)
y_test = np.ascontiguousarray(y_mm[test_idx], dtype=np.int64)

# In-place standardization: no second full-size feature copy.
X_train -= scaler_mean
X_train /= scaler_scale
X_test -= scaler_mean
X_test /= scaler_scale

if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
    raise ValueError("Non-finite values found after scaling.")

# Unmap the source arrays and free split indices; client_positions are relative to X_train.
del X_mm, y_mm, train_idx, test_idx
gc.collect()

materialize_seconds = time.perf_counter() - materialize_start
print("Materialized training samples:", f"{len(y_train):,}")
print("Materialized testing samples: ", f"{len(y_test):,}")
print("Materialize + scale time:", f"{materialize_seconds:.2f} s")
show_ram("After materialization")


In [ ]:
# Cell 08: Create tensors and choose training-data location
# Test data stays on CPU. Training data moves to GPU once if it safely fits,
# exactly like the optimized centralized baseline.
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

train_bytes = X_train.nbytes + y_train.nbytes
move_train_to_gpu = False

if KEEP_TRAIN_ON_GPU and device.type == "cuda":
    free_bytes, _ = torch.cuda.mem_get_info()
    move_train_to_gpu = train_bytes < GPU_DATA_MAX_FRACTION * free_bytes

if move_train_to_gpu:
    print(f"Moving {train_bytes/1024**3:.2f} GB of training data to GPU once...")
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    train_data_device = device
else:
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    train_data_device = torch.device("cpu")

# Tensor objects now own/reference everything needed.
del X_train, y_train, X_test, y_test
gc.collect()

print("Training data location:", train_data_device)
print("Test data location: CPU")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Cell 09: Define the exact centralized-baseline MLP
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_1=128, hidden_2=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, num_classes),
        )

    def forward(self, x):
        return self.network(x)


global_model = MLP(
    input_dim=len(feature_cols),
    num_classes=len(classes),
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
).to(device)

local_model = MLP(
    input_dim=len(feature_cols),
    num_classes=len(classes),
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
).to(device)

num_parameters = sum(p.numel() for p in global_model.parameters())
amp_enabled = bool(USE_AMP and device.type == "cuda")

print(global_model)
print("Trainable parameters:", f"{num_parameters:,}")
print("AMP enabled:", amp_enabled)


In [ ]:
# Cell 10: Loss function
if USE_CLASS_WEIGHTS:
    train_counts = np.bincount(
        y_train_t.detach().cpu().numpy(),
        minlength=len(classes),
    ).astype(np.float64)
    if np.any(train_counts == 0):
        raise ValueError("A class has zero training samples.")
    class_weights_np = len(y_train_t) / (len(classes) * train_counts)
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

print("Class weights:", USE_CLASS_WEIGHTS)
print("Loss: CrossEntropyLoss")


In [ ]:
# Cell 11: Sanity check one batch before expensive FL training
n = min(BATCH_SIZE, len(client_positions[0]))
positions = torch.from_numpy(client_positions[0][:n].astype(np.int64, copy=False)).to(train_data_device)
X_batch = X_train_t[positions]
y_batch = y_train_t[positions]

if train_data_device.type == "cpu":
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

local_model.eval()
with torch.no_grad():
    ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
    with ctx:
        logits = local_model(X_batch)

print("X:", X_batch.shape, X_batch.dtype, X_batch.device)
print("y:", y_batch.shape, y_batch.dtype, y_batch.device)
print("Output:", logits.shape)
print("Finite outputs:", bool(torch.isfinite(logits).all()))

assert logits.shape == (n, len(classes))
assert torch.isfinite(logits).all()

del positions, X_batch, y_batch, logits
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Cell 12: Local client training
# A client starts each round from the SAME current global weights,
# trains only on its own IID partition, then returns its updated model parameters.
def train_one_client(global_state, client_id, round_number):
    positions_np = client_positions[client_id]

    # Reset local model to the current global model.
    local_model.load_state_dict(global_state, strict=True)
    local_model.train()

    optimizer = torch.optim.Adam(local_model.parameters(), lr=LEARNING_RATE)
    grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    running_loss = 0.0
    correct = 0
    total = 0
    client_start = time.perf_counter()

    # Client positions live on the same device as the training tensors.
    positions_t = torch.from_numpy(positions_np.astype(np.int64, copy=False)).to(train_data_device)

    for local_epoch in range(1, LOCAL_EPOCHS + 1):
        # Deterministic but different shuffle for each round/client/local epoch.
        shuffle_seed = SEED + round_number * 100_000 + client_id * 1_000 + local_epoch
        generator = torch.Generator(device=train_data_device)
        generator.manual_seed(shuffle_seed)
        order = torch.randperm(len(positions_t), generator=generator, device=train_data_device)

        num_batches = (len(positions_t) + BATCH_SIZE - 1) // BATCH_SIZE

        for batch_number, start in enumerate(range(0, len(positions_t), BATCH_SIZE), start=1):
            batch_positions = positions_t[order[start:start + BATCH_SIZE]]
            X_batch = X_train_t[batch_positions]
            y_batch = y_train_t[batch_positions]

            if train_data_device.type == "cpu":
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

            optimizer.zero_grad(set_to_none=True)

            ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
            with ctx:
                logits = local_model(X_batch)
                loss = criterion(logits, y_batch)

            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()

            batch_n = y_batch.size(0)
            running_loss += loss.item() * batch_n
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            total += batch_n

            if batch_number % 250 == 0 or batch_number == num_batches:
                elapsed = time.perf_counter() - client_start
                print(
                    f"  client {client_id:02d} | local epoch {local_epoch}/{LOCAL_EPOCHS} | "
                    f"batch {batch_number:,}/{num_batches:,} | "
                    f"loss {running_loss/total:.4f} | acc {100*correct/total:.2f}% | "
                    f"{total/elapsed:,.0f} samples/s"
                )

        del order

    elapsed = time.perf_counter() - client_start
    local_state_cpu = {
        key: value.detach().cpu().clone()
        for key, value in local_model.state_dict().items()
    }

    del positions_t, optimizer, grad_scaler

    return {
        "state": local_state_cpu,
        "samples": int(len(positions_np)),
        "loss": float(running_loss / total),
        "accuracy": float(correct / total),
        "seconds": float(elapsed),
        "samples_per_second": float(total / elapsed),
    }


In [ ]:
# Cell 13: One complete FedAvg round with streaming weighted aggregation
# Model states are tiny here (~15.6k parameters), but streaming aggregation keeps the logic clean
# and avoids storing all client models at once.
def run_fedavg_round(round_number):
    round_start = time.perf_counter()

    # Snapshot current global weights. Every client starts from this exact state.
    global_state = {
        key: value.detach().clone()
        for key, value in global_model.state_dict().items()
    }

    total_samples = sum(len(x) for x in client_positions)
    aggregated_state = None
    client_rows = []

    for client_id in range(NUM_CLIENTS):
        print(f"\nRound {round_number}/{NUM_ROUNDS} — training client {client_id}/{NUM_CLIENTS - 1}")
        result = train_one_client(global_state, client_id, round_number)
        weight = result["samples"] / total_samples

        if aggregated_state is None:
            aggregated_state = {
                key: tensor * weight
                for key, tensor in result["state"].items()
            }
        else:
            for key in aggregated_state:
                aggregated_state[key].add_(result["state"][key], alpha=weight)

        client_rows.append({
            "round": round_number,
            "client_id": client_id,
            "samples": result["samples"],
            "train_loss": result["loss"],
            "train_accuracy": result["accuracy"],
            "seconds": result["seconds"],
            "samples_per_second": result["samples_per_second"],
        })

        print(
            f"Client {client_id:02d} complete | samples={result['samples']:,} | "
            f"loss={result['loss']:.4f} | acc={100*result['accuracy']:.2f}% | "
            f"time={result['seconds']:.2f}s"
        )

        del result
        gc.collect()

    # FedAvg result becomes the new global model.
    global_model.load_state_dict(aggregated_state, strict=True)

    round_seconds = time.perf_counter() - round_start
    client_df = pd.DataFrame(client_rows)

    summary = {
        "round": round_number,
        "client_train_loss": float(np.average(client_df["train_loss"], weights=client_df["samples"])),
        "client_train_accuracy": float(np.average(client_df["train_accuracy"], weights=client_df["samples"])),
        "round_training_seconds": float(round_seconds),
        "client_mean_samples_per_second": float(client_df["samples_per_second"].mean()),
    }

    del global_state, aggregated_state
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return summary, client_df


In [ ]:
# Cell 14: Global evaluation — same headline metrics as the centralized baseline
def evaluate_global_model(model):
    model.eval()
    predictions = np.empty(len(y_test_t), dtype=np.int16)
    targets = np.empty(len(y_test_t), dtype=np.int16)

    start_time = time.perf_counter()
    write_pos = 0

    with torch.no_grad():
        for start in range(0, len(y_test_t), EVAL_BATCH_SIZE):
            X_batch = X_test_t[start:start + EVAL_BATCH_SIZE].to(device)
            y_batch = y_test_t[start:start + EVAL_BATCH_SIZE].to(device)

            ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
            with ctx:
                logits = model(X_batch)

            pred = logits.argmax(dim=1)
            n = len(y_batch)
            predictions[write_pos:write_pos + n] = pred.cpu().numpy()
            targets[write_pos:write_pos + n] = y_batch.cpu().numpy()
            write_pos += n

    evaluation_seconds = time.perf_counter() - start_time

    accuracy = accuracy_score(targets, predictions)
    balanced_accuracy = balanced_accuracy_score(targets, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        targets, predictions, average="macro", zero_division=0
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        targets, predictions, average="weighted", zero_division=0
    )
    mcc = matthews_corrcoef(targets, predictions)
    cohen_kappa = cohen_kappa_score(targets, predictions)

    metrics = {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_precision),
        "weighted_recall": float(weighted_recall),
        "weighted_f1": float(weighted_f1),
        "mcc": float(mcc),
        "cohen_kappa": float(cohen_kappa),
        "evaluation_seconds": float(evaluation_seconds),
    }

    return metrics, targets, predictions


In [ ]:
# Cell 15: Train multiple federated rounds, evaluate after every round, and checkpoint
round_history = []
client_history_records = []
start_round = 1

if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    global_model.load_state_dict(checkpoint["global_model_state_dict"])
    round_history = checkpoint.get("round_history", [])
    client_history_records = checkpoint.get("client_history", [])
    start_round = int(checkpoint.get("round", 0)) + 1
    print(f"Resuming from completed FL round {start_round - 1}.")

training_session_start = time.perf_counter()

for round_number in range(start_round, NUM_ROUNDS + 1):
    round_summary, client_df = run_fedavg_round(round_number)

    print(f"\nEvaluating global model after round {round_number}...")
    metrics, _, _ = evaluate_global_model(global_model)
    round_summary.update(metrics)
    round_history.append(round_summary)
    client_history_records.extend(client_df.to_dict(orient="records"))

    torch.save({
        "round": round_number,
        "global_model_state_dict": global_model.state_dict(),
        "round_history": round_history,
        "client_history": client_history_records,
        "config": {
            "num_rounds": NUM_ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "seed": SEED,
            "num_clients": NUM_CLIENTS,
        },
        "classes": classes,
        "feature_cols": feature_cols,
        "prepared_metadata": metadata,
    }, CHECKPOINT_PATH)

    print("\nRound summary:")
    for key, value in round_summary.items():
        if isinstance(value, float):
            print(f"{key:<32} {value:.6f}")
        else:
            print(f"{key:<32} {value}")
    print("Checkpoint saved:", CHECKPOINT_PATH)

training_session_seconds = time.perf_counter() - training_session_start
round_history_df = pd.DataFrame(round_history)
client_history_df = pd.DataFrame(client_history_records)

print("\nFederated round history:")
display(round_history_df)
print("This session training time:", f"{training_session_seconds:.2f} s")


In [ ]:
# Cell 16: Final global evaluation and per-class metrics
final_metrics, test_targets, test_predictions = evaluate_global_model(global_model)

print("Final global-model metrics:")
for name, value in final_metrics.items():
    if name != "evaluation_seconds":
        print(f"{name:<24} {value:.4f}")
print("Evaluation time:", f"{final_metrics['evaluation_seconds']:.2f} s")

report_dict = classification_report(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

per_class_df = pd.DataFrame([
    {
        "class_id": i,
        "class_name": name,
        "precision": report_dict[name]["precision"],
        "recall": report_dict[name]["recall"],
        "f1": report_dict[name]["f1-score"],
        "support": int(report_dict[name]["support"]),
    }
    for i, name in enumerate(classes)
])

display(per_class_df.sort_values("f1"))


In [ ]:
# Cell 17: Convergence plots
if not round_history_df.empty:
    plt.figure(figsize=(7, 4))
    plt.plot(round_history_df["round"], round_history_df["client_train_loss"], marker="o")
    plt.xlabel("Federated round")
    plt.ylabel("Weighted client training loss")
    plt.title("FedAvg IID — Training Loss by Round")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fedavg_iid_train_loss_by_round.png", dpi=160)
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(round_history_df["round"], round_history_df["macro_f1"], marker="o", label="Macro F1")
    plt.plot(round_history_df["round"], round_history_df["accuracy"], marker="o", label="Accuracy")
    plt.xlabel("Federated round")
    plt.ylabel("Score")
    plt.title("FedAvg IID — Global Test Performance")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fedavg_iid_global_metrics_by_round.png", dpi=160)
    plt.show()


In [ ]:
# Cell 18: Normalized confusion matrix — same style of final diagnostic as centralized baseline
cm_norm = confusion_matrix(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    normalize="true",
)

plt.figure(figsize=(18, 16))
plt.imshow(cm_norm, interpolation="nearest", aspect="auto")
plt.colorbar()
plt.xticks(np.arange(len(classes)), classes, rotation=90, fontsize=7)
plt.yticks(np.arange(len(classes)), classes, fontsize=7)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("FedAvg IID — Normalized Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fedavg_iid_confusion_matrix_normalized.png", dpi=180)
plt.show()


In [ ]:
# Cell 19: Comparison-ready result row and experiment summary
final_round = int(round_history_df["round"].max()) if not round_history_df.empty else 0

total_round_training_seconds = (
    float(round_history_df["round_training_seconds"].sum())
    if not round_history_df.empty else 0.0
)

comparison_metrics_df = pd.DataFrame([{
    "experiment": "FedAvg-IID",
    "dataset_frac": float(metadata["dataset_frac_requested"]),
    "train_samples": int(len(y_train_t)),
    "test_samples": int(len(y_test_t)),
    "num_clients": NUM_CLIENTS,
    "rounds": final_round,
    "local_epochs": LOCAL_EPOCHS,
    "effective_data_passes": final_round * LOCAL_EPOCHS,
    "accuracy": final_metrics["accuracy"],
    "balanced_accuracy": final_metrics["balanced_accuracy"],
    "macro_precision": final_metrics["macro_precision"],
    "macro_recall": final_metrics["macro_recall"],
    "macro_f1": final_metrics["macro_f1"],
    "weighted_f1": final_metrics["weighted_f1"],
    "mcc": final_metrics["mcc"],
    "cohen_kappa": final_metrics["cohen_kappa"],
}])

display(comparison_metrics_df.round(4))

experiment_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "federated_strategy": "FedAvg",
    "partition": "IID",
    "prepared_artifact": metadata,
    "model": {
        "architecture": f"{len(feature_cols)}-{HIDDEN_1}-{HIDDEN_2}-{len(classes)}",
        "trainable_parameters": int(num_parameters),
    },
    "training": {
        "num_clients": NUM_CLIENTS,
        "rounds": final_round,
        "local_epochs": LOCAL_EPOCHS,
        "effective_data_passes": final_round * LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "all_clients_per_round": True,
        "use_amp": amp_enabled,
        "training_data_location": str(train_data_device),
    },
    "metrics": {k: float(v) for k, v in final_metrics.items()},
    "round_history": round_history,
    "runtime_seconds": {
        "materialize_and_scale": float(materialize_seconds),
        "round_training_total": float(total_round_training_seconds),
        "this_session": float(training_session_seconds),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    },
}


In [ ]:
# Cell 20: Save all result artifacts
results_path = OUTPUT_DIR / "fedavg_iid_results.json"
round_history_path = OUTPUT_DIR / "fedavg_iid_round_history.csv"
client_history_path = OUTPUT_DIR / "fedavg_iid_client_history.csv"
per_class_path = OUTPUT_DIR / "fedavg_iid_per_class.csv"
comparison_path = OUTPUT_DIR / "fedavg_iid_comparison_metrics.csv"

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, indent=2)

round_history_df.to_csv(round_history_path, index=False)
client_history_df.to_csv(client_history_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
comparison_metrics_df.to_csv(comparison_path, index=False)

print("Saved result artifacts:")
for path in [
    CHECKPOINT_PATH,
    results_path,
    round_history_path,
    client_history_path,
    per_class_path,
    comparison_path,
]:
    print("-", path)


In [ ]:
# Cell 21: Package FedAvg result outputs for download
# The prepared multi-GB dataset is NOT included; only experiment results/checkpoint/plots are zipped.
zip_path = Path("/kaggle/working/fedavg_iid_all_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.iterdir():
        if file.is_file():
            z.write(file, arcname=file.name)

print("Created:", zip_path)
print("Zip size:", f"{zip_path.stat().st_size / 1024**2:.2f} MB")


## What to use next

The main value to carry forward is the **final global model after round 3** plus `fedavg_iid_comparison_metrics.csv` and `fedavg_iid_round_history.csv`.

Do not start the non-IID experiment by changing this notebook in place. Keep this IID run frozen as your federated baseline, then make a separate non-IID preparation/experiment notebook so the comparison remains reproducible.
